# 03 — ConvNeXt-Tiny with Auxiliary Ordinal Learning and Deferred Class Weighting

**Research question.** Can ordinal supervision and delayed class reweighting improve five-grade validation performance under the same official ConvNeXt-Tiny initialization? This is an exploratory integrated-first pilot, not an established novel or clinically validated method. [E5 source and protocol](../novelty_integrated/e5/README.md) · [APTOS result provenance](../results/APTOS2019/e5_integrated/README.md) · [audit](../docs/RESEARCH_CONSOLIDATION_AUDIT.md).

**Evidence boundary:** APTOS integrated has completed 20 training epochs, best validation epoch 16, and a separate one-time frozen final test. DDR integrated is **TRAINING_IN_PROGRESS** on HPC (operator report: epoch 8 complete, epoch 9 underway); local committed final summary is unavailable. DDR final test is **FINAL_TEST_NOT_RUN**. Control, ordinal-only and deferred-only arms are implemented but have no committed results.

## 1. Context, source, datasets and licensing

The backbone is official pinned [`facebook/convnext-tiny-224`](https://huggingface.co/facebook/convnext-tiny-224), initialized from the same verified snapshot per arm. APTOS2019 images require access under [Kaggle competition terms](https://www.kaggle.com/competitions/aptos2019-blindness-detection/data); DDR comes from [OIA-DDR](https://github.com/nkicsl/DDR-dataset). Images, frozen manifests, checkpoint binaries and cache are external to Git. Training and validation use fixed seed-42 manifests and image hashes. The source runner excludes frozen test rows from development. APTOS final-test aggregate evidence is displayed only *after* the completed training record; it must never guide model selection or reruns.

## 2. Safe environment setup

Set `DR_RESEARCH_ROOT`, `DR_ARTIFACT_ROOT`, `DR_OUTPUT_ROOT` to accessible paths. Colab can mount Drive explicitly; HPC should use the pinned environment. This cell checks dependency availability and device only. No download, caching, GPU job, or output write happens. Results inspection remains CPU-only.

In [ ]:
from pathlib import Path
import os, sys, importlib.util, hashlib, json, csv
from IPython.display import display, Markdown

def show_table(rows, columns):
    if not rows:
        print("No recorded rows available")
        return
    header = "| " + " | ".join(columns) + " |"
    rule = "| " + " | ".join("---" for _ in columns) + " |"
    values = []
    for row in rows:
        values.append("| " + " | ".join(f"{row.get(k, '—'):.4f}" if isinstance(row.get(k), float)
                                      else str(row.get(k, '—')) for k in columns) + " |")
    display(Markdown("\n".join([header, rule, *values])))

def find_root():
    configured = os.environ.get("DR_RESEARCH_ROOT")
    if configured:
        root = Path(configured).expanduser().resolve()
        if not (root / "README.md").is_file():
            raise FileNotFoundError(f"DR_RESEARCH_ROOT is not a repository: {root}")
        return root
    for root in (Path.cwd(), *Path.cwd().parents):
        if (root / "README.md").is_file() and (root / "novelty_integrated" / "e5").is_dir():
            return root.resolve()
    raise FileNotFoundError("Set DR_RESEARCH_ROOT to the cloned repository")

ROOT = find_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
ARTIFACT_ROOT = Path(os.environ.get("DR_ARTIFACT_ROOT", ROOT.parent / "Kel_13")).expanduser().resolve()
OUTPUT_ROOT = Path(os.environ.get("DR_OUTPUT_ROOT", ROOT.parent / "research_new_runs")).expanduser().resolve()

def verified_example(dataset, image_path, split):
    """Verify a display image is exactly a frozen train/validation manifest row."""
    if split not in {"train", "val"}:
        raise ValueError("Only train/val examples are permitted")
    manifest = ARTIFACT_ROOT / "cad_vit_runs" / "manifests" / f"{dataset}_seed42.csv"
    if not manifest.is_file() or not manifest.with_suffix(".json").is_file():
        return False
    if hashlib.sha256(manifest.read_bytes()).hexdigest() != MANIFEST_SHA256[dataset]:
        raise ValueError("Frozen manifest hash mismatch")
    target = Path(image_path).expanduser().resolve()
    root = (ARTIFACT_ROOT / "datasets" / dataset).resolve()
    with manifest.open(encoding="utf-8", newline="") as stream:
        for row in csv.DictReader(stream):
            if row["split"] == "test":
                continue  # do not inspect test path, ID, label or hash
            if row["split"] != split:
                continue
            candidate = (root / row["image_path"]).resolve()
            if root not in candidate.parents:
                raise ValueError("Manifest path escapes dataset root")
            if candidate == target:
                if hashlib.sha256(target.read_bytes()).hexdigest() != row["sha256"]:
                    raise ValueError("Example image hash mismatch")
                print("Verified frozen", dataset, split, "example ID:", row["image_id"])
                return row["image_id"]
    raise ValueError("Example is not a frozen train/validation row; test access refused")
RUN_PREFLIGHT = False
RUN_MODEL_SANITY = False
RUN_SMOKE = False
RUN_TRAINING = False
RUN_RESUME = False
print("repository:", ROOT)
print("external artifacts:", ARTIFACT_ROOT, "available:", ARTIFACT_ROOT.is_dir())
print("external output (no directory created):", OUTPUT_ROOT)
print("Python:", sys.version.split()[0])
print({name: importlib.util.find_spec(name) is not None for name in ("torch", "torchvision", "transformers", "PIL", "cv2", "numpy", "pandas", "matplotlib")})
if importlib.util.find_spec("torch"):
    import torch
    print("torch:", torch.__version__, "CUDA:", torch.cuda.is_available())
else:
    print("SKIPPED_MISSING_DATA: PyTorch-dependent model cells require the pinned project environment")

### Optional Colab Drive mount and frozen manifest identity

Mounting is disabled by default. Manifest SHA checks run only when external CSV/sidecar files exist; the source E5 runner performs stronger image, group and processor checks before any smoke or training.

In [ ]:
# Run only when using Colab and after obtaining authorized dataset access.
MOUNT_DRIVE = False
if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    print("Set DR_ARTIFACT_ROOT and DR_OUTPUT_ROOT, then rerun the setup cell")
else:
    print("Drive mount skipped; no packages or weights downloaded")

The next cell checks the original frozen CSV bytes without opening any image. If manifests are absent in a fresh clone, it says `SKIPPED_MISSING_DATA`.

In [ ]:
from pathlib import Path
MANIFEST_SHA256 = {"APTOS2019": "73d1546e3c2fd2bf9e3d0fd10af914cb523d28e4bc8e1f25212af454d4be8854",
                   "DDR": "7bc3e3916e012bc46fcd9cd69aa469e0daf3229e1273f1d8ecb6054899bf6eb1"}
for dataset, expected in MANIFEST_SHA256.items():
    csv_path = ARTIFACT_ROOT / "cad_vit_runs" / "manifests" / f"{dataset}_seed42.csv"
    sidecar = csv_path.with_suffix(".json")
    if not csv_path.is_file() or not sidecar.is_file():
        print(dataset, "SKIPPED_MISSING_DATA: frozen CSV/sidecar not present on this host")
        continue
    actual = hashlib.sha256(csv_path.read_bytes()).hexdigest()
    assert actual == expected, f"{dataset} manifest SHA mismatch"
    print(dataset, "frozen CSV SHA-256 verified; sidecar present:", sidecar)
    print("For training, use the source runner's full image/group/hash preflight")

## 3. EDA and actual-image examples

Class support comes from the recorded APTOS E5 training run, not an inferred split. The next cell plots train and validation grade counts from `run.json`, using no patient images. Re-running is safe.

In [ ]:
try:
    import matplotlib.pyplot as plt
except ImportError:
    plt = None
    print("SKIPPED_MISSING_DATA: matplotlib unavailable; committed figures remain visible")
E5_DIR = ROOT / "results/APTOS2019/e5_integrated"
run = json.loads((E5_DIR / "run.json").read_text(encoding="utf-8"))
print("dataset, arm, epoch budget, seed:", run["dataset"], run["arm"], run["epochs"], run["seed"])
print("recorded train counts:", run["train_counts"])
print("manifest metadata keys:", list(run["manifest"]))
counts = run["train_counts"]
if isinstance(counts, dict):
    keys = sorted(counts, key=int); values = [counts[k] for k in keys]
else:
    keys = list(range(len(counts))); values = counts
if plt is not None:
    plt.bar(keys, values); plt.xlabel("DR grade"); plt.ylabel("Training images")
    plt.title("APTOS E5 frozen training support"); plt.show()

A real-image before/after view requires an authorized **train or validation** image supplied by the operator. E5 uses EXIF-transposed RGB, B1 conservative augmentation on training only, then the pinned official processor at 224 px. The next cell only shows the original RGB sample; exact processed tensors and stochastic augmentation require the pinned snapshot and the source input-pipeline verifier. It never accesses a test sample or invents a gallery.

In [ ]:
sample = os.environ.get("DR_SAMPLE_IMAGE")
split = os.environ.get("DR_SAMPLE_SPLIT", "")
SAMPLE_VERIFIED = False
SAMPLE_ROW_ID = None
if not sample:
    print("SKIPPED_MISSING_DATA: set DR_SAMPLE_IMAGE and DR_SAMPLE_SPLIT=train or val")
elif split not in {"train", "val"}:
    print("SKIPPED_MISSING_DATA: test images are excluded from development examples")
elif not (SAMPLE_ROW_ID := verified_example("APTOS2019", sample, split)):
    print("SKIPPED_MISSING_DATA: frozen APTOS manifest and sidecar unavailable for image verification")
else:
    SAMPLE_VERIFIED = True
    from PIL import Image, ImageOps
    with Image.open(sample) as im:
        display(ImageOps.exif_transpose(im).convert("RGB").resize((224, 224)))
    print("Use E5 verify-inputs for exact processor/cache equivalence after snapshot and manifest checks")

### Actual official processing and B1 augmentation example

With an authorized train/validation sample, its manifest image ID for training, and the original official snapshot, the next cell verifies snapshot and processor hashes before creating the real 224×224 tensor. It calls the source B1 augmentation only for training, then displays the processor input and denormalized output. This is CPU preprocessing only; it does not load the model, run inference, or write files. Exact cache equivalence is a separate `verify-inputs` gate.

In [ ]:
snapshot_dir = os.environ.get("DR_CONVNEXT_SNAPSHOT", "")
if not SAMPLE_VERIFIED or not snapshot_dir:
    print("SKIPPED_MISSING_DATA: train/val sample or pinned official snapshot unavailable")
else:
    from novelty import foundation_adaptation as fa
    from PIL import Image, ImageOps
    import numpy as np
    expected = run["snapshot"]
    audited = fa.audit_snapshot("convnext_tiny", snapshot_dir, expected["revision"], expected["weight_sha256"])
    assert audited["processor_config_sha256"] == expected["processor_config_sha256"]
    processor = fa.load_local_processor(audited)
    with Image.open(sample) as im:
        rgb = ImageOps.exif_transpose(im).convert("RGB")
    image_id = SAMPLE_ROW_ID
    if split == "train" and image_id:
        from novelty import foundation_augmentation
        augmented, _ = foundation_augmentation.apply(rgb, seed=42, epoch=1, image_id=image_id)
        display(rgb.resize((224, 224)), augmented.resize((224, 224)))
        processed = processor(images=augmented, return_tensors="pt")["pixel_values"]
    else:
        print("No train image ID: deterministic official processing only")
        processed = processor(images=rgb, return_tensors="pt")["pixel_values"]
    mean = np.asarray(processor.image_mean)[:, None, None]
    std = np.asarray(processor.image_std)[:, None, None]
    array = np.clip(processed[0].numpy() * std + mean, 0, 1).transpose(1, 2, 0)
    display(Image.fromarray((array * 255).round().astype(np.uint8)))
    print("official tensor:", tuple(processed.shape), processed.dtype)

## 4. Architecture and mathematical objective

The official ConvNeXt-Tiny backbone produces pooled features `h`. The five-class categorical head computes `z=W_c h+b_c`, and **only `argmax(z)`** determines the reported DR grade. The auxiliary severity head computes `s=w_oᵀh+b_o`; ordered thresholds satisfy `τ₀=a`, `τₖ=τₖ₋₁+softplus(dₖ)+10⁻⁴` for k=1,2,3. Ordinal targets are `1[y>k]` with logits `s−τₖ`. Threshold order is enforced by construction, and all labels come from training grade only.

The training objective is `mean_i[w_epoch(y_i) CE(z_i,y_i)] + 0.25 mean_{i,k} BCEWithLogits(s_i−τ_k,1[y_i>k])`. The second term is absent in control/deferred. For deferred/integrated at epoch ≥11, `w_c = n_c^(−1/2) / (Σ_j n_j n_j^(−1/2)/N_train)` using training counts. Epochs 1–10 use ordinary CE. This is an auxiliary ordinal adaptation, **not** [Stelter et al.'s distribution-based loss](https://proceedings.mlr.press/v307/stelter26a.html) or a novel mechanism established by this study. See [E5 implementation](../novelty_integrated/e5/model.py), [objective](../novelty_integrated/e5/objective.py) and [protocol](../novelty_integrated/e5/README.md).

This diagram follows the implemented `model.py` and `objective.py`: both heads read the same pooled feature, while reported grades come exclusively from the categorical head.

```mermaid
flowchart LR
    I[Official 224px processor] --> B[Pinned ConvNeXt-Tiny backbone]
    B --> F[768-dimensional pooled feature]
    F --> C[Five-class categorical head]
    F --> O[Severity and ordered thresholds]
    C --> P[Reported grade by argmax]
    C --> L[Categorical CE]
    O --> A[Auxiliary ordinal BCE]
    L --> T[Total training loss]
    A --> T
```

### Recorded parameter counts and source-model sanity

`summary.json` stores actual parameters from the constructed run: backbone 27,820,128 and total trainable 27,824,746. The next cell verifies this recorded count and prints the pinned processor metadata. A fresh model sanity check or loaded checkpoint requires the external official snapshot and is guarded below; no weights are downloaded here.

In [ ]:
summary = json.loads((E5_DIR / "summary.json").read_text(encoding="utf-8"))
print("Actual recorded run parameter counts:", summary["parameter_counts"])
print("Official processor:", run["effective_processor"])
print("Pinned snapshot metadata:", run["snapshot"])
assert summary["parameter_counts"]["total_trainable"] == 27824746

## 5. Four controlled arms and optimization

All arms use official pretrained initialization, seed 42, full fine-tuning, B1 training-only augmentation, AdamW (backbone LR 1e−5, head LR 1e−3, weight decay .01), BF16 when supported, effective batch 32 and a 20-epoch pilot. The source [configuration](../novelty_integrated/e5/config.py) and [runner](../novelty_integrated/e5/runner.py) lock the exact details and protect resume. Best checkpoint maximizes **validation macro-F1**, then chooses lower validation categorical CE, then earlier epoch. Validation/test use deterministic official preprocessing. The arms below differ only in objective components.

| Arm | Ordinal auxiliary | Delayed class weights | Recorded outcome |
| --- | --- | --- | --- |
| `control` | No | No | NOT RUN in committed E5 evidence |
| `ordinal` | Yes | No | NOT RUN in committed E5 evidence |
| `deferred` | No | Yes, epochs 11–20 | NOT RUN in committed E5 evidence |
| `integrated` | Yes | Yes, epochs 11–20 | APTOS complete; DDR TRAINING_IN_PROGRESS |

The four-arm design separates effects of ordinal supervision and weighting; it is required before attributing any gain to their combination. Existing B1/B0–B4 validation data are historical context, not an independently matched E5 control.

### Source preflight, smoke, cache and training interface (all disabled)

The next cell builds the exact E5 CLI with externally supplied pinned snapshot and SHA. `prepare-cache` can create an isolated **validation-only deterministic tensor cache** after its own integrity checks; stochastic training augmentation is never cached. `verify-inputs` checks equivalence; `smoke` checks model and optimizer; `train` requires a matching smoke report. This notebook does not run any command by default. Keep every output outside Git and away from completed APTOS/active DDR directories.

In [ ]:
import subprocess
DATASET = "APTOS2019"  # switch to DDR only with its own frozen manifest and a NEW run directory
ARM = "integrated"  # control / ordinal / deferred / integrated
MODEL_DIR = os.environ.get("DR_CONVNEXT_SNAPSHOT", "")
MODEL_REVISION = os.environ.get("DR_CONVNEXT_REVISION", "")
MODEL_WEIGHT_SHA256 = os.environ.get("DR_CONVNEXT_WEIGHT_SHA256", "")
SMOKE_REPORT = os.environ.get("DR_E5_SMOKE_REPORT", "")
NEW_RUN = OUTPUT_ROOT / "E5" / DATASET / (ARM + "_new")
SMOKE_DIR = OUTPUT_ROOT / "E5" / DATASET / (ARM + "_smoke_new")
if not SMOKE_REPORT:
    SMOKE_REPORT = str(SMOKE_DIR / "smoke.json")
e5_base = [sys.executable, "-m", "novelty_integrated.e5.runner",
           "--dataset", DATASET, "--arm", ARM, "--artifact-root", str(ARTIFACT_ROOT),
           "--model-dir", MODEL_DIR, "--revision", MODEL_REVISION,
           "--weight-sha256", MODEL_WEIGHT_SHA256]
def e5_command(action):
    destination = SMOKE_DIR if action == "smoke" else NEW_RUN
    return [sys.executable, "-m", "novelty_integrated.e5.runner", action] + e5_base[3:] + ["--output-dir", str(destination)]
print("Smoke command:", e5_command("smoke"))
print("Train command:", e5_command("train") + ["--smoke-report", SMOKE_REPORT])
if RUN_SMOKE:
    if not all((MODEL_DIR, MODEL_REVISION, MODEL_WEIGHT_SHA256)) or not ARTIFACT_ROOT.is_dir():
        print("SKIPPED_MISSING_DATA: pinned model or external frozen artifacts unavailable")
    elif ROOT in SMOKE_DIR.parents or SMOKE_DIR.exists():
        raise RuntimeError("Use a new external E5 smoke directory; do not reuse historical output")
    else:
        subprocess.run(e5_command("smoke"), cwd=ROOT, check=True)
else:
    print("SKIPPED: RUN_SMOKE=False")

### Explicit guarded train/resume

The next cell delegates the actual train/validation loop, metrics, checkpoint hashes, `last.pt`/`best.pt`, optimizer/RNG resume, and CSV/JSON logging to the source runner. It is disabled until `RUN_TRAINING` and either fresh `RUN_SMOKE` or reviewed `RUN_RESUME` are enabled, an external snapshot is configured, and a matching smoke report exists. A smoke report is stored in a **separate directory** from the training run. Resume uses the source runner's guarded `--resume --trust-run-checkpoint` path and should only target a new run created by this workflow. Never aim at the active DDR or completed APTOS run. This cell does **not** expose a final-test command.

In [ ]:
if RUN_TRAINING and (RUN_SMOKE or RUN_RESUME):
    if not SMOKE_REPORT or not Path(SMOKE_REPORT).is_file() or ROOT in NEW_RUN.parents:
        raise RuntimeError("Verified smoke report and external output are required")
    if RUN_RESUME and not NEW_RUN.exists():
        raise RuntimeError("Cannot resume a nonexistent E5 run")
    if not RUN_RESUME and NEW_RUN.exists():
        raise RuntimeError("New E5 output already exists; choose a fresh run identity")
    resume_flags = ["--resume", "--trust-run-checkpoint"] if RUN_RESUME else []
    subprocess.run(e5_command("train") + ["--smoke-report", SMOKE_REPORT] + resume_flags,
                   cwd=ROOT, check=True)
else:
    print("SKIPPED: training disabled; completed APTOS and active DDR outputs remain untouched")

## 6. APTOS integrated: actual 20-epoch training and validation

The 20 rows in `history.csv` are read directly. The code shows epoch-wise train/validation losses, accuracy and macro-F1, weighted-F1, QWK, grade MAE, class-weight activation, and best epoch. It does not rerun training or evaluate any image. `summary.json.final_test_status=NOT_ACCESSED` is the truthful end-of-training snapshot; the subsequent final test is held in a separate file.

In [ ]:
import csv
with (E5_DIR / "history.csv").open(encoding="utf-8", newline="") as handle:
    history = list(csv.DictReader(handle))
assert len(history) == 20 and [int(h["epoch"]) for h in history] == list(range(1, 21))
columns = ["epoch", "train_total_loss", "train_categorical_loss", "train_ordinal_loss",
           "train_accuracy", "train_macro_f1", "validation_categorical_ce", "validation_ordinal_loss",
           "validation_accuracy", "validation_macro_f1", "validation_weighted_f1",
           "validation_qwk", "validation_grade_mae", "class_weight_active", "best_epoch"]
show_table(history, columns)
print("Best epoch:", summary["best"]["epoch"],
      "validation macro-F1:", summary["best"]["validation"]["macro_f1"],
      "end-of-training test status:", summary["final_test_status"])

**All 20 original E5 APTOS epoch records, selected fields from `history.csv` (no imputation).**

| Epoch | Train total loss | Train categorical | Train ordinal | Train accuracy | Train macro-F1 | Val CE | Val accuracy | Val macro-F1 | Val weighted-F1 | Val QWK | Val grade MAE | Weights active |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| 1 | 0.897157 | 0.817037 | 0.320482 | 0.704437 | 0.360525 | 0.624256 | 0.759563 | 0.428665 | 0.712516 | 0.788277 | 0.341530 | False |
| 2 | 0.608571 | 0.547844 | 0.242909 | 0.792491 | 0.496780 | 0.564878 | 0.765027 | 0.444230 | 0.723281 | 0.811341 | 0.322404 | False |
| 3 | 0.530770 | 0.474720 | 0.224201 | 0.822526 | 0.613118 | 0.518513 | 0.792350 | 0.549204 | 0.767044 | 0.858902 | 0.273224 | False |
| 4 | 0.484593 | 0.430499 | 0.216375 | 0.831399 | 0.654159 | 0.494170 | 0.816940 | 0.633014 | 0.806823 | 0.887604 | 0.237705 | False |
| 5 | 0.449483 | 0.397669 | 0.207254 | 0.843686 | 0.674938 | 0.511709 | 0.811475 | 0.613454 | 0.797466 | 0.881523 | 0.240437 | False |
| 6 | 0.415148 | 0.365363 | 0.199139 | 0.863481 | 0.726790 | 0.490705 | 0.814208 | 0.635035 | 0.803980 | 0.887492 | 0.234973 | False |
| 7 | 0.382118 | 0.334039 | 0.192314 | 0.875768 | 0.753091 | 0.492732 | 0.819672 | 0.664214 | 0.816649 | 0.893126 | 0.229508 | False |
| 8 | 0.351648 | 0.305131 | 0.186070 | 0.886007 | 0.776812 | 0.477212 | 0.822404 | 0.660030 | 0.817217 | 0.897007 | 0.224044 | False |
| 9 | 0.320469 | 0.275210 | 0.181037 | 0.895222 | 0.796010 | 0.486822 | 0.825137 | 0.681788 | 0.825638 | 0.889836 | 0.232240 | False |
| 10 | 0.282479 | 0.239357 | 0.172492 | 0.913652 | 0.825426 | 0.490415 | 0.819672 | 0.669234 | 0.819188 | 0.899166 | 0.224044 | False |
| 11 | 0.345273 | 0.303376 | 0.167587 | 0.923891 | 0.857947 | 0.539067 | 0.814208 | 0.692351 | 0.819259 | 0.892512 | 0.243169 | True |
| 12 | 0.313007 | 0.271940 | 0.164267 | 0.931058 | 0.867483 | 0.515008 | 0.833333 | 0.699700 | 0.834340 | 0.906091 | 0.210383 | True |
| 13 | 0.273173 | 0.233155 | 0.160069 | 0.945051 | 0.899154 | 0.536960 | 0.838798 | 0.704220 | 0.839084 | 0.907110 | 0.204918 | True |
| 14 | 0.241536 | 0.202575 | 0.155843 | 0.955973 | 0.918755 | 0.525128 | 0.816940 | 0.676927 | 0.819374 | 0.896254 | 0.234973 | True |
| 15 | 0.219791 | 0.181886 | 0.151620 | 0.957679 | 0.924821 | 0.551838 | 0.827869 | 0.701311 | 0.830779 | 0.897218 | 0.226776 | True |
| 16 | 0.183858 | 0.146987 | 0.147488 | 0.968942 | 0.942157 | 0.588535 | 0.830601 | 0.705791 | 0.833080 | 0.899241 | 0.221311 | True |
| 17 | 0.162890 | 0.126903 | 0.143949 | 0.975085 | 0.953548 | 0.580192 | 0.827869 | 0.702080 | 0.830590 | 0.895021 | 0.229508 | True |
| 18 | 0.139545 | 0.105100 | 0.137781 | 0.979181 | 0.960507 | 0.593483 | 0.803279 | 0.661122 | 0.807395 | 0.882819 | 0.265027 | True |
| 19 | 0.131551 | 0.098171 | 0.133519 | 0.979522 | 0.960072 | 0.628240 | 0.819672 | 0.688347 | 0.821982 | 0.890958 | 0.240437 | True |
| 20 | 0.121527 | 0.088727 | 0.131200 | 0.976451 | 0.955747 | 0.646363 | 0.822404 | 0.694021 | 0.825716 | 0.894043 | 0.234973 | True |

### Stored learning curves

The actual 20-epoch learning curves are visible on GitHub without a kernel. Validation macro-F1 peaks at epoch 16, whereas training loss continues to decline; this alone does not prove the auxiliary term improves generalization. Compare the other matched E5 arms when available.

![APTOS E5 recorded training and validation learning curves](../results/APTOS2019/e5_integrated/learning_curves.svg)

### Validation metrics, per-grade errors and confusion matrix

The next cell reads the validation-selected checkpoint metrics from `summary.json`. The five-grade confusion matrix has 366 rows in total and 304 correct. Grade 1 and grade 3 require explicit review because aggregate macro-F1 can hide low recall. This is an aggregate read only.

In [ ]:
v = summary["best"]["validation"]
assert sum(map(sum, v["confusion_matrix"])) == 366 and sum(v["confusion_matrix"][i][i] for i in range(5)) == 304
print({k: v[k] for k in ("accuracy", "macro_f1", "weighted_f1", "quadratic_weighted_kappa", "grade_mae", "correct", "wrong")})
show_table(v["per_grade"], ["grade", "support", "correct", "precision", "recall", "f1", "mean_absolute_error"])
show_table([{**{"true_grade": i}, **{f"pred_{j}": count for j, count in enumerate(row)}}
            for i, row in enumerate(v["confusion_matrix"])],
           ["true_grade", "pred_0", "pred_1", "pred_2", "pred_3", "pred_4"])

**Best-validation per-grade evidence from `summary.json` (epoch 16).**

| Grade | Support | Correct | Precision | Recall | F1 |
| --- | --- | --- | --- | --- | --- |
| 0 | 180 | 174 | 0.994286 | 0.966667 | 0.980282 |
| 1 | 35 | 26 | 0.553191 | 0.742857 | 0.634146 |
| 2 | 101 | 76 | 0.775510 | 0.752475 | 0.763819 |
| 3 | 20 | 10 | 0.588235 | 0.500000 | 0.540541 |
| 4 | 30 | 18 | 0.620690 | 0.600000 | 0.610169 |

The committed figure uses true grades as rows and predictions as columns. It is a validation result, not a final-test matrix.

![APTOS E5 best-validation confusion matrix](../results/APTOS2019/e5_integrated/validation_confusion_matrix.svg)

## 7. APTOS integrated: locked one-time frozen final test

`test_summary.json` is a separate post-training record tied to checkpoint epoch 16. The original training `summary.json` remains `NOT_ACCESSED`; the distinct test file is `FINAL_TEST_COMPLETE`. This cell reads committed aggregate results only; it does not load images, predictions, or weights. The 366 test rows include 15 additional occurrences of duplicate SHA/group IDs **within the test split**, with no train/validation overlap detected in the existing audit; all 366 remain counted.

In [ ]:
test = json.loads((E5_DIR / "test_summary.json").read_text(encoding="utf-8"))
assert test["status"] == "FINAL_TEST_COMPLETE" and test["test_samples"] == 366
assert test["checkpoint_epoch"] == summary["best"]["epoch"] == 16
t = test["metrics"]
assert sum(map(sum, t["confusion_matrix"])) == 366 and sum(t["confusion_matrix"][i][i] for i in range(5)) == 302
print("One-time final test:", {k: t[k] for k in ("accuracy", "macro_f1", "weighted_f1", "quadratic_weighted_kappa", "grade_mae", "correct", "wrong")})
show_table(t["per_grade"], ["grade", "support", "correct", "precision", "recall", "f1", "mean_absolute_error"])
show_table([{**{"true_grade": i}, **{f"pred_{j}": count for j, count in enumerate(row)}}
            for i, row in enumerate(t["confusion_matrix"])],
           ["true_grade", "pred_0", "pred_1", "pred_2", "pred_3", "pred_4"])

**One-time final-test per-grade evidence from `test_summary.json`.**

| Grade | Support | Correct | Precision | Recall | F1 |
| --- | --- | --- | --- | --- | --- |
| 0 | 180 | 176 | 0.994350 | 0.977778 | 0.985994 |
| 1 | 38 | 25 | 0.694444 | 0.657895 | 0.675676 |
| 2 | 100 | 83 | 0.721739 | 0.830000 | 0.772093 |
| 3 | 19 | 4 | 0.307692 | 0.210526 | 0.250000 |
| 4 | 29 | 14 | 0.560000 | 0.482759 | 0.518519 |

The frozen final-test matrix is a separate original figure. Grade 3 has 19 examples, 4 correct and F1=.25; errors concentrate toward adjacent grades in the aggregate matrix. This pattern does not establish a clinical cause. Test macro-F1=.640456 versus selected validation macro-F1=.705791 indicates a generalization gap; do not tune on this locked test result.

![APTOS E5 one-time final-test confusion matrix](../results/APTOS2019/e5_integrated/test_confusion_matrix.svg)

## 8. DDR E5 status and future artifact loading

Operator status on 20 November 2026: DDR integrated training is **TRAINING_IN_PROGRESS** on HPC (`run_retry1`; epoch 8 complete, epoch 9 underway at the last report). No completed 20-epoch summary or legitimate DDR E5 final-test result is available in this checkout. The cell below only checks an **optional external report path**; it never opens active checkpoint binaries or assumes that training finished. Supply `DR_E5_DDR_SUMMARY` after the run completes and is audited. Until then: `FINAL_TEST_NOT_RUN`.

In [ ]:
ddr_report = os.environ.get("DR_E5_DDR_SUMMARY", "")
if not ddr_report or not Path(ddr_report).is_file():
    print("DDR E5: TRAINING_IN_PROGRESS; completed summary SKIPPED_MISSING_DATA; FINAL_TEST_NOT_RUN")
else:
    ddr = json.loads(Path(ddr_report).read_text(encoding="utf-8"))
    if ddr.get("epochs_completed") != 20 or ddr.get("status") not in {"VALIDATION_COMPLETE", "COMPLETED"}:
        print("DDR E5: source summary does not establish completed 20-epoch validation; no final metrics displayed")
    else:
        print("DDR E5 completion artifact found; verify manifest/checkpoint provenance before publishing metrics")

## 9. Scientific interpretation and limits

APTOS integrated's best validation macro-F1 (.705791) is close to the historical B1 (.701124) and below B4-R (.707897); those are different training protocols and do not identify the integrated mechanism's effect. Matched `control`, `ordinal`, and `deferred` arms under E5 are needed. Grade-3 final-test F1=.25 and only 4/19 correct caution against clinical claims. One seed, repeated validation use, dataset imbalance, ordered-label supervision overlap, preprocessing dependence, and no external clinical validation limit interpretation. The APTOS final test is locked and must not be rerun for tuning; DDR final test has not run. No unsupported novelty or state-of-the-art claim is made.